# 7.12 — ConvNeXt

ConvNeXt is a modern convolutional backbone: it keeps the CNN promise that local kernels find reusable spatial patterns, but it borrows transformer-era habits — larger kernels, LayerNorm-style scaling, inverted channel expansion, and clean residual corrections — so a convolutional network trains and scales like a contemporary vision model.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build ConvNeXt one idea at a time. Run each cell in order and read the printed intermediate values — every piece of math is spelled out so the block is not a black box. This walkthrough is self-contained (it imports what it needs) and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays and explicit convolution loops.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for small weights.

### 1. Depthwise convolution: local evidence per channel

ConvNeXt does not replace convolution with attention. Its spatial mixer is a **depthwise convolution**: each channel gets its own spatial kernel, scans nearby pixels, and produces one response map for that same channel. The tiny arithmetic is the same as any convolution: multiply a local patch by a kernel and sum. With patch $\begin{bmatrix}1&2\\3&4\end{bmatrix}$ and diagonal kernel $\begin{bmatrix}1&0\\0&-1\end{bmatrix}$, the response is $1\cdot1+2\cdot0+3\cdot0+4\cdot(-1)=-3$.

In [ ]:
patch_w = np.array([[1., 2.], [3., 4.]])  # one local spatial window from one channel.
kernel_w = np.array([[1., 0.], [0., -1.]])  # a simple diagonal contrast filter.
products_w = patch_w * kernel_w  # elementwise evidence before summing.
print("patch × kernel:\n", products_w)  # inspect the four products.
print("depthwise response:", products_w.sum())  # 1 + 0 + 0 - 4 = -3.
assert products_w.sum() == -3.0  # concrete number from the lesson block.

▶ What you'll see: only the diagonal entries contribute, giving a response of `-3`.

In [ ]:
plt.figure(figsize=(6, 2.6))  # compare input, kernel, and products.
for idx_w, (mat_w, title_w) in enumerate([(patch_w, "patch"), (kernel_w, "kernel"), (products_w, "products")]):
    plt.subplot(1, 3, idx_w + 1)  # one small heatmap per object.
    plt.imshow(mat_w, cmap="coolwarm", vmin=-4, vmax=4)  # signed colors show positive/negative evidence.
    plt.title(title_w)  # label the panel.
    plt.xticks([]); plt.yticks([])  # remove ticks so the arithmetic is the focus.
plt.suptitle("1: one depthwise response = sum(patch × kernel)"); plt.show()

▶ What you'll see: the product heatmap makes the `+1` and `−4` contributions visible.

*Why it's done this way:* Depthwise convolution preserves the convolutional inductive bias — nearby pixels matter together — while avoiding full channel mixing during the spatial step. In a full convolution, every output channel combines every input channel and every spatial location; in a depthwise convolution, channel $c$ only applies its own kernel to channel $c$. That separation is the ConvNeXt design bet: first collect local evidence cheaply inside each channel, then let pointwise layers mix channel meanings afterward.

### 2. Sliding kernels make feature maps, not single numbers

A convolutional backbone is useful because the same local test is reused across the grid. Sliding the same diagonal kernel over a ramp image produces a whole feature map. On $\begin{bmatrix}1&2&3\\4&5&6\\7&8&9\end{bmatrix}$, every valid $2\times2$ window has the same diagonal difference: top-left is $1-5=-4$, top-right is $2-6=-4$, and the bottom windows match.

In [ ]:
image_w = np.array([[1., 2., 3.], [4., 5., 6.], [7., 8., 9.]])  # a tiny ramp image.
out_w = np.zeros((2, 2))  # valid 2x2 convolution output shape.
for i_w in range(2):  # slide down.
    for j_w in range(2):  # slide right.
        window_w = image_w[i_w:i_w+2, j_w:j_w+2]  # current local patch.
        out_w[i_w, j_w] = np.sum(window_w * kernel_w)  # same kernel at every location.
print("feature map:\n", out_w)  # every local slope is the same.
assert np.all(out_w == -4.0)  # concrete lesson number: all four responses are -4.

▶ What you'll see: a 2×2 feature map filled with `-4`, meaning the same local slope appears everywhere.

In [ ]:
plt.figure(figsize=(6, 2.8))  # show input and response side by side.
plt.subplot(1, 2, 1); plt.imshow(image_w, cmap="viridis"); plt.title("input ramp"); plt.colorbar(fraction=.046)
plt.subplot(1, 2, 2); plt.imshow(out_w, cmap="magma"); plt.title("sliding response"); plt.colorbar(fraction=.046)
plt.suptitle("2: one kernel reused across spatial positions"); plt.show()

▶ What you'll see: the input values rise, but the response map is constant because the local diagonal contrast is constant.

*Why it's done this way:* Weight sharing is the mathematical heart of convolution. The kernel parameters are reused at all locations, so the model can learn one local pattern and recognize it anywhere. This gives translation equivariance: moving the pattern in the input moves the response in the feature map instead of requiring a new set of weights.

### 3. LayerNorm across channels at each location

ConvNeXt moves away from batch-statistics dependence and normalizes features per spatial location across channels. For one pixel with channel values $[1,2,3]$, LayerNorm computes mean $\mu=2$, variance $\sigma^2=2/3$, and normalized values approximately $[-1.225,0,1.225]$.

In [ ]:
pixel_w = np.array([1., 2., 3.])  # three channel values at one spatial position.
mu_w = pixel_w.mean()  # mean across channels only.
var_w = np.mean((pixel_w - mu_w) ** 2)  # variance across channels only.
ln_w = (pixel_w - mu_w) / np.sqrt(var_w)  # epsilon omitted here to match the hand arithmetic.
print("mean:", mu_w, "variance:", round(var_w, 3))  # 2 and 2/3.
print("LayerNorm(pixel):", np.round(ln_w, 3))  # [-1.225, 0, 1.225].
assert round(var_w, 3) == 0.667  # concrete variance.
assert np.allclose(np.round(ln_w, 3), [-1.225, 0.000, 1.225])  # concrete normalized values.

▶ What you'll see: the three channels become centered around 0 with unit-ish spread.

In [ ]:
plt.figure(figsize=(5, 2.8))  # compare raw and normalized channels.
plt.bar(np.arange(3) - 0.18, pixel_w, width=0.36, label="raw", color="gray")  # raw channel values.
plt.bar(np.arange(3) + 0.18, ln_w, width=0.36, label="LayerNorm", color="teal")  # normalized values.
plt.axhline(0, color="black", linewidth=.7)  # zero reference after normalization.
plt.xticks(range(3), ["ch0", "ch1", "ch2"]); plt.legend(); plt.title("3: normalize channels at one location"); plt.show()

▶ What you'll see: raw values sit at 1,2,3; normalized values are balanced around zero.

*Why it's done this way:* The following pointwise channel mixer should not have to chase arbitrary feature scale. Normalizing the $C$ channels at each $(h,w)$ makes the hidden channel computation see comparable magnitudes even when batches are small or images vary. The axis matters: normalizing over height and width too would mix spatial content into the statistic and change what each location means.

### 4. Inverted bottleneck: expand channels, compute, project back

After spatial mixing and normalization, ConvNeXt spends computation in channels. If a location has $C=3$ channels and expansion ratio $4$, the hidden width is $12$. The first pointwise matrix $W_1$ maps $3\to12$, a nonlinearity acts there, and $W_2$ maps $12\to3$ so the correction can be added back to the original input.

In [ ]:
C_w = 3  # original channel count.
ratio_w = 4  # ConvNeXt-style expansion ratio.
hidden_w = C_w * ratio_w  # expanded channel width.
param_count_w = C_w * hidden_w + hidden_w * C_w  # W1 plus W2 weights at one spatial location.
print("hidden width:", hidden_w)  # 12.
print("pointwise weights:", param_count_w)  # 3*12 + 12*3 = 72.
assert hidden_w == 12 and param_count_w == 72  # concrete lesson numbers.

▶ What you'll see: the block temporarily widens from 3 channels to 12 and uses 72 pointwise weights.

In [ ]:
xloc_w = np.array([0.5, -1.0, 2.0])  # one normalized-ish location vector.
W1_w = np.linspace(-0.3, 0.3, C_w * hidden_w).reshape(C_w, hidden_w)  # deterministic expand weights.
W2_w = np.linspace(0.2, -0.2, hidden_w * C_w).reshape(hidden_w, C_w)  # deterministic project weights.
h_w = np.maximum(0, xloc_w @ W1_w)  # small ReLU channel computation.
correction_w = h_w @ W2_w  # project back to C channels.
print("hidden shape:", h_w.shape, "correction shape:", correction_w.shape)  # (12,), (3,).
assert h_w.shape == (12,) and correction_w.shape == (3,)  # residual-compatible shape.

▶ What you'll see: the hidden vector is wide, but the correction returns to the original 3 channels.

In [ ]:
yloc_w = xloc_w + correction_w  # residual add after projection.
print("x:", np.round(xloc_w, 3))  # original location.
print("correction:", np.round(correction_w, 3))  # learned delta.
print("y = x + correction:", np.round(yloc_w, 3))  # residual output.
plt.figure(figsize=(5, 2.8)); plt.bar(["ch0", "ch1", "ch2"], correction_w, color="purple")
plt.axhline(0, color="black", linewidth=.7); plt.title("4: projected channel correction"); plt.show()

▶ What you'll see: the block produces a small per-channel correction that can be safely added to `x`.

*Why it's done this way:* The expansion gives the channel mixer enough room to create nonlinear combinations, but the projection restores shape. The residual equation $y=x+f(x)$ requires identical $H\times W\times C$ shapes; otherwise the add is undefined or relies on accidental broadcasting. Mathematically, residual learning makes the block learn a correction to an already-useful representation, which is easier than relearning the whole representation at every depth.

### 5. Downsampling stages: shrink space, widen meaning

ConvNeXt still behaves like a CNN backbone with stages. Between stages, strided convolutions deliberately reduce spatial resolution so later blocks can model broader semantics with more channels. The output-size arithmetic for a 1-D side length is $$\left\lfloor\frac{N+2P-K}{S}\right\rfloor+1.$$ A $5\times5$ map with a $2\times2$ kernel, stride 2, and no padding becomes $2\times2$; with padding 1 and stride 1, it becomes $6\times6$.

In [ ]:
def conv_out_size_w(N_w, K_w, S_w=1, P_w=0):  # standard convolution shape formula.
    return (N_w + 2 * P_w - K_w) // S_w + 1  # integer output side length.
size_stride_w = conv_out_size_w(5, 2, S_w=2, P_w=0)  # floor((5-2)/2)+1.
size_pad_w = conv_out_size_w(5, 2, S_w=1, P_w=1)  # floor((5+2-2)/1)+1.
print("stride-2 output side:", size_stride_w)  # 2.
print("pad-1 stride-1 output side:", size_pad_w)  # 6.
assert size_stride_w == 2 and size_pad_w == 6  # concrete lesson numbers.

▶ What you'll see: stride shrinks the map, while padding can preserve or enlarge the valid output grid.

In [ ]:
grid_w = np.arange(25).reshape(5, 5)  # toy 5x5 feature map.
down_w = grid_w[0:4:2, 0:4:2]  # show which top-left samples a stride-2 2x2 scan touches.
print("sampled top-left positions:\n", down_w)  # four positions for the 2x2 output.
plt.figure(figsize=(5.4, 2.7))
plt.subplot(1, 2, 1); plt.imshow(grid_w, cmap="viridis"); plt.title("5×5 stage input"); plt.colorbar(fraction=.046)
plt.subplot(1, 2, 2); plt.imshow(down_w, cmap="viridis"); plt.title("2×2 stride positions"); plt.colorbar(fraction=.046)
plt.suptitle("5: stage downsampling changes the grid deliberately"); plt.show()

▶ What you'll see: the coarse grid keeps fewer spatial positions, preparing the network to spend more computation per location.

*Why it's done this way:* Early layers need fine grids to capture edges and textures; deeper layers need wider context and semantic channels. Downsampling is the controlled exchange rate: fewer spatial positions make later channel-rich computation affordable, while the stage hierarchy keeps a feature pyramid that detection and segmentation heads can use.

## 🛠️ Setup

In [ ]:
import numpy as np  # load NumPy for arrays, loops, normalization, and assertions.
import matplotlib.pyplot as plt  # load Matplotlib for all heatmaps, bars, and curves.
np.random.seed(0)  # make the examples reproducible.

def conv2d_valid(x, k):  # compute a single-channel valid 2-D convolution/correlation from scratch.
    H, W = x.shape  # read input height and width.
    KH, KW = k.shape  # read kernel height and width.
    out = np.zeros((H - KH + 1, W - KW + 1))  # allocate valid output grid.
    for i in range(out.shape[0]):  # slide vertically.
        for j in range(out.shape[1]):  # slide horizontally.
            out[i, j] = np.sum(x[i:i+KH, j:j+KW] * k)  # multiply patch by kernel and sum.
    return out  # return the feature map.

def depthwise_conv_valid(x, kernels):  # apply one spatial kernel per channel to an HxWxC tensor.
    H, W, C = x.shape  # read grid shape and channel count.
    KH, KW, CK = kernels.shape  # read kernel shape.
    assert C == CK  # each channel must have its own kernel.
    out = np.zeros((H - KH + 1, W - KW + 1, C))  # allocate one output map per channel.
    for c in range(C):  # loop over channels independently.
        out[:, :, c] = conv2d_valid(x[:, :, c], kernels[:, :, c])  # convolve only that channel.
    return out  # return depthwise feature maps.

def layer_norm_channels(x, eps=1e-6):  # normalize each spatial location across channels.
    mu = np.mean(x, axis=-1, keepdims=True)  # channel mean per location.
    var = np.mean((x - mu) ** 2, axis=-1, keepdims=True)  # channel variance per location.
    return (x - mu) / np.sqrt(var + eps)  # normalized tensor.

def relu(x):  # simple nonlinearity for pointwise channel mixing.
    return np.maximum(0, x)  # keep positive values and zero negatives.

def pointwise(x, W):  # apply a 1x1 linear map independently at every location.
    return x @ W  # last dimension C is multiplied by W.

def convnext_tiny_block(x, kernels, W1, W2):  # depthwise conv, LayerNorm, expansion, projection, residual crop.
    z = depthwise_conv_valid(x, kernels)  # spatial mixing per channel.
    z = layer_norm_channels(z)  # normalize channels at each output location.
    h = relu(pointwise(z, W1))  # expand and apply nonlinearity.
    corr = pointwise(h, W2)  # project back to original channel count.
    return x[:corr.shape[0], :corr.shape[1], :] + corr  # residual add on matching spatial crop.

def conv_out_size(N, K, S=1, P=0):  # compute convolution output side length.
    return (N + 2 * P - K) // S + 1  # standard integer formula.

def show_tensor_channel(x, c, title):  # small helper to visualize one channel of a tensor.
    plt.figure(figsize=(3.5, 3))  # compact figure.
    plt.imshow(x[:, :, c], cmap="viridis", aspect="auto")  # heatmap the selected channel.
    plt.colorbar(label="value")  # numeric color scale.
    plt.title(title)  # label the figure.
    plt.show()  # display it.

## 🟢 Basics (warm-up)

### Basic 1 — Multiply one patch by one kernel

**Goal.** Compute the primitive convolution response, because every ConvNeXt spatial mixer starts from local patch evidence. We build it in 2 steps.

In [ ]:
patch_b1 = np.array([[1., 2.], [3., 4.]])  # define one local image patch.
kernel_b1 = np.array([[1., 0.], [0., -1.]])  # define a diagonal contrast kernel.
print("patch:\n", patch_b1)  # inspect the data window.
print("kernel:\n", kernel_b1)  # inspect the filter weights.

In [ ]:
response_b1 = float(np.sum(patch_b1 * kernel_b1))  # multiply corresponding entries and sum.
print("response:", response_b1)  # inspect the scalar response.
assert response_b1 == -3.0  # verify 1 - 4 = -3.
plt.figure(figsize=(4, 3))  # create a product heatmap.
plt.imshow(patch_b1 * kernel_b1, cmap="coolwarm")  # show positive and negative contributions.
plt.colorbar(label="product")  # add a scale.
plt.title("Basic 1: patch × kernel")  # title the diagnostic plot.
plt.show()  # display the figure.

▶ What you'll see: the products sum to `-3`, with the bottom-right negative term dominating.

👀 Takeaway: convolution is local weighted summation before it becomes a deep-network block.

### Basic 2 — Slide a kernel over a small image

**Goal.** Turn one response into a feature map, because a convolutional filter is reused across spatial positions. We build it in 2 steps.

In [ ]:
image_b2 = np.array([[1., 2., 3.], [4., 5., 6.], [7., 8., 9.]])  # define a 3x3 ramp.
kernel_b2 = np.array([[1., 0.], [0., -1.]])  # reuse the diagonal contrast filter.
print("image shape:", image_b2.shape, "kernel shape:", kernel_b2.shape)  # inspect valid output setup.

In [ ]:
feat_b2 = conv2d_valid(image_b2, kernel_b2)  # slide the kernel and collect responses.
print("feature map:\n", feat_b2)  # inspect all local responses.
assert np.all(feat_b2 == -4.0)  # verify the ramp produces constant diagonal contrast.
plt.figure(figsize=(4, 3))  # create a compact feature-map plot.
plt.imshow(feat_b2, cmap="magma")  # visualize responses.
plt.colorbar(label="response")  # add response scale.
plt.title("Basic 2: sliding response map")  # title the plot.
plt.show()  # display it.

▶ What you'll see: every valid window gives `-4`, so the output is a constant 2×2 map.

👀 Takeaway: weight sharing lets one local test detect the same pattern anywhere.

### Basic 3 — Apply depthwise convolution to two channels

**Goal.** Use one kernel per channel, because ConvNeXt separates spatial mixing from channel mixing. We build it in 3 steps.

In [ ]:
x_b3 = np.dstack([image_b2, image_b2 + 10])  # create a 3x3x2 tensor with two channels.
kernels_b3 = np.dstack([kernel_b2, -kernel_b2])  # give each channel its own spatial filter.
print("input shape:", x_b3.shape, "kernels shape:", kernels_b3.shape)  # inspect channel alignment.

In [ ]:
dw_b3 = depthwise_conv_valid(x_b3, kernels_b3)  # convolve each channel independently.
print("depthwise output shape:", dw_b3.shape)  # valid 2x2 grid with two channels.
print("channel means:", np.round(dw_b3.mean(axis=(0, 1)), 3))  # inspect one summary per channel.
assert dw_b3.shape == (2, 2, 2)  # verify depthwise preserves the channel count.

In [ ]:
plt.figure(figsize=(5, 2.6))  # visualize both output channels.
for c_b3 in range(2):  # loop over channels.
    plt.subplot(1, 2, c_b3 + 1)  # one panel per channel.
    plt.imshow(dw_b3[:, :, c_b3], cmap="coolwarm", vmin=-4, vmax=4)  # signed depthwise responses.
    plt.title(f"channel {c_b3}")  # label channel.
    plt.colorbar(fraction=.046)  # add compact colorbar.
plt.suptitle("Basic 3: one spatial filter per channel"); plt.show()

▶ What you'll see: channel 0 responds with `-4` while channel 1 responds with `+4` because its kernel is negated.

👀 Takeaway: depthwise convolution mixes space inside each channel but does not mix channels together.

### Basic 4 — Count depthwise versus full convolution weights

**Goal.** Compare parameter counts, because depthwise convolution is cheap enough to use larger kernels like 7×7. We build it in 2 steps.

In [ ]:
C_b4 = 8  # number of input and output channels for a same-width comparison.
K_b4 = 7  # ConvNeXt-style large spatial kernel.
depthwise_params_b4 = K_b4 * K_b4 * C_b4  # one KxK kernel per channel.
full_params_b4 = K_b4 * K_b4 * C_b4 * C_b4  # every output channel mixes every input channel spatially.
print("depthwise params:", depthwise_params_b4)  # 7*7*8.
print("full conv params:", full_params_b4)  # 7*7*8*8.
assert depthwise_params_b4 == 392 and full_params_b4 == 3136  # concrete counts.

In [ ]:
plt.figure(figsize=(4, 3))  # create a parameter comparison.
plt.bar(["depthwise", "full"], [depthwise_params_b4, full_params_b4], color=["teal", "crimson"])  # compare counts.
plt.title("Basic 4: 7×7 parameter cost")  # title the plot.
plt.ylabel("weights")  # label the count axis.
plt.show()  # display it.

▶ What you'll see: full convolution uses 8× more weights in this same-width example.

👀 Takeaway: depthwise kernels broaden local context without the full channel-mixing cost.

### Basic 5 — Normalize channels at one pixel

**Goal.** Compute LayerNorm for one location, because ConvNeXt normalizes across channels before channel mixing. We build it in 2 steps.

In [ ]:
pixel_b5 = np.array([1., 2., 3.])  # one spatial position with three channels.
mean_b5 = pixel_b5.mean()  # channel mean.
var_b5 = np.mean((pixel_b5 - mean_b5) ** 2)  # channel variance.
print("mean:", mean_b5, "variance:", round(var_b5, 3))  # inspect the statistics.
assert round(var_b5, 3) == 0.667  # verify 2/3 variance.

In [ ]:
ln_b5 = (pixel_b5 - mean_b5) / np.sqrt(var_b5)  # normalize across channels.
print("normalized:", np.round(ln_b5, 3))  # inspect normalized values.
assert np.allclose(np.round(ln_b5, 3), [-1.225, 0.000, 1.225])  # verify lesson values.
plt.figure(figsize=(4, 3))  # compare channel values.
plt.bar(["ch0", "ch1", "ch2"], ln_b5, color="seagreen")  # visualize normalized channels.
plt.axhline(0, color="black", linewidth=.7)  # zero reference.
plt.title("Basic 5: per-location LayerNorm")  # title the plot.
plt.show()  # display it.

▶ What you'll see: normalized channels are centered and scaled to comparable magnitude.

👀 Takeaway: LayerNorm makes the channel mixer see stable scales at each spatial position.

### Basic 6 — Expand channels with a pointwise layer

**Goal.** Map $C$ channels to $4C$ channels, because ConvNeXt uses an inverted bottleneck to spend computation in channel space. We build it in 2 steps.

In [ ]:
x_b6 = np.array([0.5, -1.0, 2.0])  # one location with C=3 channels.
W1_b6 = np.arange(36, dtype=float).reshape(3, 12) / 100.0  # deterministic 3-to-12 pointwise weights.
hidden_b6 = x_b6 @ W1_b6  # expand channels at one location.
print("hidden shape:", hidden_b6.shape)  # C=3 expands to 12.
assert hidden_b6.shape == (12,)  # verify expansion ratio 4.

In [ ]:
plt.figure(figsize=(5, 3))  # visualize expanded hidden channels.
plt.bar(range(12), hidden_b6, color="purple")  # show all hidden activations.
plt.title("Basic 6: 3 → 12 channel expansion")  # title the plot.
plt.xlabel("hidden channel")  # label hidden index.
plt.ylabel("activation")  # label activation value.
plt.show()  # display it.

▶ What you'll see: one 3-channel vector becomes 12 hidden channel values.

👀 Takeaway: pointwise expansion gives the block a wide channel workspace after spatial mixing.

### Basic 7 — Project back for a residual add

**Goal.** Return hidden channels to the original channel count, because the residual path requires matching shapes. We build it in 3 steps.

In [ ]:
hidden_b7 = np.maximum(0, hidden_b6)  # reuse the expanded vector after a simple nonlinearity.
W2_b7 = np.linspace(0.2, -0.2, 36).reshape(12, 3)  # deterministic 12-to-3 projection weights.
correction_b7 = hidden_b7 @ W2_b7  # project back to C=3.
print("correction shape:", correction_b7.shape)  # must match x_b6.
assert correction_b7.shape == x_b6.shape  # residual-compatible.

In [ ]:
y_b7 = x_b6 + correction_b7  # add the learned correction to the input.
print("input:", np.round(x_b6, 3))  # inspect original.
print("output:", np.round(y_b7, 3))  # inspect residual result.

In [ ]:
plt.figure(figsize=(4, 3))  # visualize input versus output.
plt.plot(x_b6, marker="o", label="x")  # original channels.
plt.plot(y_b7, marker="s", label="x + correction")  # residual output.
plt.xticks(range(3), ["ch0", "ch1", "ch2"]); plt.legend(); plt.title("Basic 7: residual-compatible projection"); plt.show()

▶ What you'll see: the output has the same three channels as the input but with a learned correction.

👀 Takeaway: ConvNeXt can add residuals cleanly because the projection returns to $C$ channels.

### Basic 8 — Compute convolution output size

**Goal.** Use the shape formula, because ConvNeXt stages deliberately change spatial resolution. We build it in 2 steps.

In [ ]:
side_b8 = 5  # input side length.
kernel_b8 = 2  # kernel side length.
stride_b8 = 2  # downsampling stride.
out_b8 = conv_out_size(side_b8, kernel_b8, S=stride_b8, P=0)  # floor((5-2)/2)+1.
print("output side:", out_b8)  # inspect the downsampled side length.
assert out_b8 == 2  # verify lesson arithmetic.

In [ ]:
out_pad_b8 = conv_out_size(5, 2, S=1, P=1)  # padded stride-1 case.
print("padded output side:", out_pad_b8)  # inspect padded size.
assert out_pad_b8 == 6  # verify lesson arithmetic.
plt.figure(figsize=(4, 3))  # compare shape cases.
plt.bar(["stride2 no pad", "stride1 pad1"], [out_b8, out_pad_b8], color=["orange", "teal"])  # show output sides.
plt.ylabel("output side length")  # label side length.
plt.title("Basic 8: convolution shape arithmetic")  # title the plot.
plt.xticks(rotation=15)  # keep labels readable.
plt.show()  # display it.

▶ What you'll see: stride shrinks the grid to side 2, while padding with stride 1 gives side 6.

👀 Takeaway: stage resolution changes are deterministic consequences of kernel, stride, and padding.

### Basic 9 — Build a tiny ConvNeXt-style block

**Goal.** Chain depthwise convolution, LayerNorm, expansion, projection, and residual add, because that is the compact ConvNeXt block recipe. We build it in 3 steps.

In [ ]:
x_b9 = np.dstack([np.arange(16).reshape(4, 4), np.arange(16, 32).reshape(4, 4)]).astype(float) / 10.0  # 4x4x2 input.
kernels_b9 = np.dstack([np.ones((2, 2)) / 4, -np.ones((2, 2)) / 4])  # one averaging/sign kernel per channel.
W1_b9 = np.ones((2, 4)) * 0.1  # expand 2 channels to 4 hidden channels.
W2_b9 = np.array([[0.2, -0.1], [0.1, 0.2], [-0.1, 0.1], [0.05, -0.05]])  # project 4 back to 2.
print("input shape:", x_b9.shape)  # inspect starting tensor.

In [ ]:
y_b9 = convnext_tiny_block(x_b9, kernels_b9, W1_b9, W2_b9)  # run the tiny block.
print("output shape:", y_b9.shape)  # valid depthwise conv creates a 3x3 residual crop.
assert y_b9.shape == (3, 3, 2)  # verify the residual output shape.

In [ ]:
show_tensor_channel(y_b9, 0, "Basic 9: output channel 0")  # visualize one output channel.

▶ What you'll see: a 3×3 output channel after local mixing and residual correction.

👀 Takeaway: ConvNeXt is a disciplined sequence of familiar operations, not a mystery layer.

### Basic 10 — Contrast local convolution with global averaging

**Goal.** Show that depthwise convolution is local rather than global attention, because ConvNeXt broadens context but still uses a finite kernel window. We build it in 2 steps.

In [ ]:
img_b10 = np.zeros((5, 5))  # create a sparse input.
img_b10[2, 2] = 10.0  # put one bright signal in the center.
local_kernel_b10 = np.ones((3, 3)) / 9.0  # local averaging kernel.
local_b10 = conv2d_valid(img_b10, local_kernel_b10)  # local responses see the signal only in nearby windows.
global_b10 = np.full_like(local_b10, img_b10.mean())  # global average broadcasts the same value everywhere.
print("local max:", round(local_b10.max(), 3), "global value:", round(global_b10[0, 0], 3))  # compare local vs global.

In [ ]:
plt.figure(figsize=(6, 2.6))  # show local and global summaries.
plt.subplot(1, 2, 1); plt.imshow(local_b10, cmap="magma"); plt.title("local 3×3 conv"); plt.colorbar(fraction=.046)
plt.subplot(1, 2, 2); plt.imshow(global_b10, cmap="magma"); plt.title("global average"); plt.colorbar(fraction=.046)
plt.suptitle("Basic 10: local context is not global attention"); plt.show()

▶ What you'll see: local convolution creates a concentrated response, while global averaging is uniform everywhere.

👀 Takeaway: a larger kernel gives broader local evidence, but not all-to-all token interaction.

## 🟡 Easy

### Easy 1 — Run depthwise 7×7-style local mixing

**Goal.** Simulate a larger depthwise kernel, because ConvNeXt widens the local window while keeping one filter per channel. We build it in 3 steps.

In [ ]:
x_e1 = np.zeros((9, 9, 2))  # create a 9x9 two-channel feature grid.
x_e1[4, 4, 0] = 1.0  # center impulse in channel 0.
x_e1[2:7, 2:7, 1] = 1.0  # square signal in channel 1.
kernels_e1 = np.dstack([np.ones((7, 7)) / 49.0, np.eye(7) / 7.0])  # average kernel and diagonal kernel.
print("input shape:", x_e1.shape, "kernel shape:", kernels_e1.shape)  # inspect 7x7 depthwise setup.

In [ ]:
y_e1 = depthwise_conv_valid(x_e1, kernels_e1)  # apply one 7x7 spatial kernel per channel.
print("output shape:", y_e1.shape)  # valid 9x9 with 7x7 becomes 3x3.
print("channel sums:", np.round(y_e1.sum(axis=(0, 1)), 3))  # inspect channel-wise evidence.
assert y_e1.shape == (3, 3, 2)  # verify output shape.

In [ ]:
plt.figure(figsize=(5, 2.6))  # visualize both depthwise outputs.
for c_e1 in range(2):  # loop over output channels.
    plt.subplot(1, 2, c_e1 + 1)  # create panel.
    plt.imshow(y_e1[:, :, c_e1], cmap="viridis")  # show channel map.
    plt.title(f"depthwise ch{c_e1}")  # label channel.
    plt.colorbar(fraction=.046)  # compact colorbar.
plt.suptitle("Easy 1: 7×7 local mixing per channel"); plt.show()

▶ What you'll see: two different response maps, because each channel owns a different 7×7 filter.

👀 Takeaway: ConvNeXt increases receptive field locally without paying full convolution channel cost.

### Easy 2 — Verify LayerNorm axis choice on a grid

**Goal.** Normalize each location across channels and compare with a wrong global normalization, because axis choice changes feature meaning. We build it in 3 steps.

In [ ]:
x_e2 = np.array([[[1., 2., 3.], [10., 20., 30.]], [[2., 2., 2.], [4., 5., 9.]]])  # 2x2x3 feature grid.
ln_e2 = layer_norm_channels(x_e2)  # correct per-location channel normalization.
global_e2 = (x_e2 - x_e2.mean()) / x_e2.std()  # wrong demo: one statistic for all space and channels.
print("input shape:", x_e2.shape)  # inspect tensor shape.

In [ ]:
means_e2 = ln_e2.mean(axis=-1)  # per-location means after correct LayerNorm.
vars_e2 = ln_e2.var(axis=-1)  # per-location variances after correct LayerNorm.
print("LN means per location:\n", np.round(means_e2, 6))  # should be near zero.
print("LN variances per location:\n", np.round(vars_e2, 3))  # constant channel locations may be near zero.
assert np.allclose(means_e2, 0, atol=1e-6)  # verify centering per location.

In [ ]:
plt.figure(figsize=(6, 2.7))  # compare correct and global normalization on channel 0.
plt.subplot(1, 2, 1); plt.imshow(ln_e2[:, :, 0], cmap="coolwarm"); plt.title("correct LN ch0"); plt.colorbar(fraction=.046)
plt.subplot(1, 2, 2); plt.imshow(global_e2[:, :, 0], cmap="coolwarm"); plt.title("global norm ch0"); plt.colorbar(fraction=.046)
plt.suptitle("Easy 2: normalization axis matters"); plt.show()

▶ What you'll see: correct LayerNorm treats each pixel independently, while global normalization mixes spatial scale into the result.

👀 Takeaway: ConvNeXt-style LayerNorm means channel normalization at each spatial position.

### Easy 3 — Compute a complete pointwise MLP at every location

**Goal.** Apply expansion and projection across a whole feature map, because ConvNeXt's channel MLP is shared over spatial positions. We build it in 3 steps.

In [ ]:
z_e3 = np.arange(18, dtype=float).reshape(3, 3, 2) / 10.0  # 3x3 grid with 2 channels.
W1_e3 = np.array([[0.5, -0.2, 0.1, 0.3], [0.4, 0.2, -0.3, 0.1]])  # 2-to-4 expansion.
W2_e3 = np.array([[0.2, -0.1], [0.1, 0.3], [-0.2, 0.2], [0.4, 0.1]])  # 4-to-2 projection.
print("z shape:", z_e3.shape)  # inspect input grid.

In [ ]:
h_e3 = relu(pointwise(z_e3, W1_e3))  # apply shared 1x1 expansion and nonlinearity.
corr_e3 = pointwise(h_e3, W2_e3)  # project hidden channels back to 2.
print("hidden shape:", h_e3.shape, "correction shape:", corr_e3.shape)  # inspect channel dimensions.
assert h_e3.shape == (3, 3, 4) and corr_e3.shape == z_e3.shape  # verify expansion/projection.

In [ ]:
plt.figure(figsize=(5, 3))  # visualize one projected correction channel.
plt.imshow(corr_e3[:, :, 0], cmap="plasma")  # show correction channel 0.
plt.colorbar(label="correction")  # add scale.
plt.title("Easy 3: pointwise correction channel 0")  # title plot.
plt.show()  # display it.

▶ What you'll see: every spatial location receives a channel correction from the same pointwise weights.

👀 Takeaway: pointwise layers mix channels locally at each pixel but share weights across the grid.

### Easy 4 — Assemble the formula y = x + W2 phi(W1 LN(DWConv(x)))

**Goal.** Run the compact ConvNeXt equation end to end, because each sub-operation must preserve the residual contract. We build it in 4 steps.

In [ ]:
x_e4 = np.dstack([np.arange(25).reshape(5, 5), np.flipud(np.arange(25).reshape(5, 5))]).astype(float) / 10.0  # 5x5x2 tensor.
kernels_e4 = np.dstack([np.ones((3, 3)) / 9.0, -np.ones((3, 3)) / 9.0])  # depthwise 3x3 kernels.
W1_e4 = np.array([[0.2, -0.1, 0.3, 0.1], [0.1, 0.4, -0.2, 0.2]])  # expand 2 to 4.
W2_e4 = np.array([[0.2, 0.1], [-0.1, 0.3], [0.4, -0.2], [0.1, 0.2]])  # project 4 to 2.
print("x shape:", x_e4.shape)  # inspect input shape.

In [ ]:
spatial_e4 = depthwise_conv_valid(x_e4, kernels_e4)  # DWConv(x).
normed_e4 = layer_norm_channels(spatial_e4)  # LN(DWConv(x)).
print("spatial shape:", spatial_e4.shape, "norm mean sample:", round(float(normed_e4[0, 0].mean()), 6))  # inspect intermediate.

In [ ]:
hidden_e4 = relu(pointwise(normed_e4, W1_e4))  # phi(W1 LN(...)).
correction_e4 = pointwise(hidden_e4, W2_e4)  # W2 projection.
y_e4 = x_e4[:3, :3, :] + correction_e4  # residual crop plus correction.
print("hidden shape:", hidden_e4.shape, "y shape:", y_e4.shape)  # inspect residual-compatible output.
assert y_e4.shape == (3, 3, 2)  # valid 3x3 output with original channel count.

In [ ]:
plt.figure(figsize=(6, 2.7))  # compare input crop and output for channel 0.
plt.subplot(1, 2, 1); plt.imshow(x_e4[:3, :3, 0], cmap="viridis"); plt.title("residual crop ch0"); plt.colorbar(fraction=.046)
plt.subplot(1, 2, 2); plt.imshow(y_e4[:, :, 0], cmap="viridis"); plt.title("ConvNeXt output ch0"); plt.colorbar(fraction=.046)
plt.suptitle("Easy 4: residual correction after modernized block"); plt.show()

▶ What you'll see: the output keeps the residual crop's shape while adding a learned channel correction.

👀 Takeaway: the ConvNeXt block is local spatial mixing plus normalized channel computation plus residual learning.

### Easy 5 — Simulate a two-stage resolution ladder

**Goal.** Downsample and widen channels, because ConvNeXt keeps the CNN stage idea of smaller grids with richer features. We build it in 3 steps.

In [ ]:
H_e5, W_e5, C_e5 = 8, 8, 2  # define a stage-1 feature shape.
stage1_e5 = np.random.default_rng(5).normal(size=(H_e5, W_e5, C_e5))  # reproducible feature grid.
print("stage 1 shape:", stage1_e5.shape)  # inspect fine grid.

In [ ]:
stage2_space_e5 = stage1_e5[::2, ::2, :]  # simple stride-2 spatial sampling for the shape demo.
W_widen_e5 = np.array([[1.0, 0.0, 0.5, -0.5], [0.0, 1.0, -0.5, 0.5]])  # pointwise 2-to-4 widening.
stage2_e5 = pointwise(stage2_space_e5, W_widen_e5)  # widen channels after downsampling.
print("stage 2 shape:", stage2_e5.shape)  # inspect coarser, wider stage.
assert stage2_e5.shape == (4, 4, 4)  # verify resolution halves and channels double.

In [ ]:
plt.figure(figsize=(5, 3))  # compare stage sizes.
plt.bar(["H×W positions", "channels"], [H_e5 * W_e5, C_e5], width=.35, label="stage1")  # fine stage counts.
plt.bar(np.arange(2) + .35, [4 * 4, 4], width=.35, label="stage2")  # coarse stage counts.
plt.xticks(np.arange(2) + .18, ["positions", "channels"]); plt.legend(); plt.title("Easy 5: downsample then widen"); plt.show()

▶ What you'll see: spatial positions drop from 64 to 16 while channels rise from 2 to 4.

👀 Takeaway: stage design trades spatial resolution for channel capacity as semantics become broader.

## 🔴 Advanced

### Advanced 1 — Compare parameter growth for kernel and channel choices

**Goal.** Sweep channel counts and compare depthwise with full convolution, because accidentally using full 7×7 convolution changes the ConvNeXt cost model. We build it in 3 steps.

In [ ]:
channels_a1 = np.array([16, 32, 64, 128])  # channel widths to compare.
K_a1 = 7  # large ConvNeXt-style kernel.
dw_params_a1 = K_a1 * K_a1 * channels_a1  # depthwise count grows linearly with channels.
full_params_a1 = K_a1 * K_a1 * channels_a1 * channels_a1  # full count grows quadratically.
print("depthwise params:", dw_params_a1)  # inspect linear growth.
print("full params:", full_params_a1)  # inspect quadratic growth.

In [ ]:
ratio_a1 = full_params_a1 / dw_params_a1  # cost multiplier from using full convolution.
print("full/depthwise ratio:", ratio_a1.astype(int))  # equals channel count for same-width conv.
assert ratio_a1[-1] == 128  # verify the largest case is 128x cost.

In [ ]:
plt.figure(figsize=(5, 3))  # log plot for parameter growth.
plt.plot(channels_a1, dw_params_a1, marker="o", label="depthwise 7×7")  # plot depthwise counts.
plt.plot(channels_a1, full_params_a1, marker="s", label="full 7×7")  # plot full counts.
plt.yscale("log")  # make both curves visible.
plt.xlabel("channels C"); plt.ylabel("parameters (log)"); plt.title("Advanced 1: full conv is a different cost regime"); plt.legend(); plt.show()

▶ What you'll see: full convolution explodes quadratically while depthwise grows linearly.

👀 Takeaway: ConvNeXt's large kernels are practical because they are depthwise, not full channel-mixing kernels.

### Advanced 2 — Show why residual shape mismatches fail

**Goal.** Demonstrate the residual constraint, because $x+f(x)$ only makes mathematical sense when shapes match. We build it in 3 steps.

In [ ]:
x_a2 = np.zeros((3, 3, 4))  # residual input with 4 channels.
correction_bad_a2 = np.ones((3, 3, 6))  # a bad projection that returns 6 channels.
print("x shape:", x_a2.shape, "bad correction shape:", correction_bad_a2.shape)  # inspect mismatch.

In [ ]:
can_add_a2 = x_a2.shape == correction_bad_a2.shape  # explicit residual shape check.
print("can add residual safely?", can_add_a2)  # should be False.
assert can_add_a2 is False  # verify mismatch is detected.

In [ ]:
correction_good_a2 = correction_bad_a2[:, :, :4]  # project/crop back to the original channel count for the demo.
y_a2 = x_a2 + correction_good_a2  # now shapes match.
print("good output shape:", y_a2.shape)  # inspect safe residual output.
assert y_a2.shape == x_a2.shape  # verify residual-compatible shape.
plt.figure(figsize=(4, 3)); plt.bar(["bad C", "good C", "x C"], [6, 4, 4], color=["crimson", "teal", "gray"])
plt.title("Advanced 2: residual channel contract"); plt.ylabel("channels"); plt.show()

▶ What you'll see: the bad correction has 6 channels, while the residual path requires exactly 4.

👀 Takeaway: the projection layer is not optional; it restores the shape needed for residual learning.

### Advanced 3 — Measure receptive field growth with stacked local kernels

**Goal.** Stack local convolutions and track which input pixels can influence the output, because larger context can come from kernel size and depth. We build it in 3 steps.

In [ ]:
support_a3 = np.zeros((11, 11))  # create an impulse support map.
support_a3[5, 5] = 1.0  # one active center pixel.
k3_a3 = np.ones((3, 3))  # a 3x3 all-ones kernel marks reachable neighbors.
k7_a3 = np.ones((7, 7))  # a 7x7 kernel marks broader local reach.
print("initial active pixels:", int(support_a3.sum()))  # one source pixel.

In [ ]:
reach3_once_a3 = conv2d_valid(np.pad(support_a3, 1), k3_a3) > 0  # same-size 3x3 reach after one layer.
reach3_twice_a3 = conv2d_valid(np.pad(reach3_once_a3.astype(float), 1), k3_a3) > 0  # reach after two 3x3 layers.
reach7_once_a3 = conv2d_valid(np.pad(support_a3, 3), k7_a3) > 0  # same-size 7x7 reach after one layer.
print("3x3 once active:", reach3_once_a3.sum(), "3x3 twice active:", reach3_twice_a3.sum(), "7x7 once active:", reach7_once_a3.sum())  # compare context.
assert reach3_once_a3.sum() == 9 and reach3_twice_a3.sum() == 25 and reach7_once_a3.sum() == 49  # concrete support sizes.

In [ ]:
plt.figure(figsize=(7, 2.5))  # visualize receptive supports.
for idx_a3, (mat_a3, title_a3) in enumerate([(reach3_once_a3, "one 3×3"), (reach3_twice_a3, "two 3×3"), (reach7_once_a3, "one 7×7")]):
    plt.subplot(1, 3, idx_a3 + 1)  # one panel per support map.
    plt.imshow(mat_a3, cmap="Greys")  # active receptive positions.
    plt.title(title_a3)  # label support.
    plt.xticks([]); plt.yticks([])  # remove ticks.
plt.suptitle("Advanced 3: local receptive field growth"); plt.show()

▶ What you'll see: one 7×7 layer covers 49 positions, while two 3×3 layers cover a 5×5 region.

👀 Takeaway: larger kernels are a direct way to broaden local evidence without global attention.

### Advanced 4 — Test normalization stability under batch shifts

**Goal.** Compare per-location LayerNorm with a batch/global statistic, because ConvNeXt avoids depending on unstable batch-scale estimates. We build it in 4 steps.

In [ ]:
batch_a4 = np.stack([np.ones((2, 2, 3)), 10 * np.ones((2, 2, 3))], axis=0)  # two images with very different brightness.
batch_a4[0, 0, 0] = np.array([1., 2., 3.])  # add channel variation to image 0.
batch_a4[1, 0, 0] = np.array([10., 20., 30.])  # same pattern scaled in image 1.
print("batch shape:", batch_a4.shape)  # B,H,W,C.

In [ ]:
ln_img0_a4 = layer_norm_channels(batch_a4[0])  # per-image, per-location channel LN.
ln_img1_a4 = layer_norm_channels(batch_a4[1])  # same for second image.
global_mu_a4 = batch_a4.mean(axis=(0, 1, 2), keepdims=True)  # batch/spatial mean per channel.
global_std_a4 = batch_a4.std(axis=(0, 1, 2), keepdims=True) + 1e-6  # batch/spatial std per channel.
global_norm_a4 = (batch_a4 - global_mu_a4) / global_std_a4  # batch-dependent normalization demo.
print("LN sample image0:", np.round(ln_img0_a4[0, 0], 3))  # scale-invariant local pattern.
print("LN sample image1:", np.round(ln_img1_a4[0, 0], 3))  # same normalized pattern.

In [ ]:
same_pattern_a4 = np.allclose(np.round(ln_img0_a4[0, 0], 3), np.round(ln_img1_a4[0, 0], 3))  # LayerNorm preserves scaled pattern.
print("LayerNorm samples match after scaling?", same_pattern_a4)  # should be True.
assert same_pattern_a4  # verify scale stability for this pattern.

In [ ]:
plt.figure(figsize=(5, 3))  # compare local LN and batch-normalized sample values.
plt.plot(ln_img0_a4[0, 0], marker="o", label="LN image0")  # local normalized pattern.
plt.plot(ln_img1_a4[0, 0], marker="s", label="LN image1")  # scaled local normalized pattern.
plt.plot(global_norm_a4[1, 0, 0], marker="^", label="batch/global image1")  # batch-stat result.
plt.xticks(range(3), ["ch0", "ch1", "ch2"]); plt.legend(); plt.title("Advanced 4: LayerNorm ignores batch brightness shift"); plt.show()

▶ What you'll see: LayerNorm gives matching channel patterns for scaled pixels, while batch/global normalization depends on the batch distribution.

👀 Takeaway: channel-wise LayerNorm stabilizes each token/location without needing reliable batch statistics.

### Advanced 5 — Compare ConvNeXt block variants on a toy signal

**Goal.** Ablate the block pieces, because ConvNeXt's behavior comes from the combination of depthwise spatial mixing, normalization, channel expansion, and residual addition. We build it in 5 steps.

In [ ]:
x_a5 = np.dstack([np.eye(6), np.fliplr(np.eye(6))]).astype(float)  # two-channel crossing-line signal.
kernels_a5 = np.dstack([np.ones((3, 3)) / 9.0, -np.ones((3, 3)) / 9.0])  # depthwise local mixers.
W1_a5 = np.array([[0.3, -0.2, 0.1, 0.4], [0.2, 0.3, -0.4, 0.1]])  # 2-to-4 expansion.
W2_a5 = np.array([[0.2, 0.1], [-0.2, 0.2], [0.1, -0.1], [0.3, 0.2]])  # 4-to-2 projection.
print("toy signal shape:", x_a5.shape)  # inspect input tensor.

In [ ]:
spatial_a5 = depthwise_conv_valid(x_a5, kernels_a5)  # depthwise spatial-only output.
no_norm_corr_a5 = pointwise(relu(pointwise(spatial_a5, W1_a5)), W2_a5)  # channel mixer without LayerNorm.
with_norm_corr_a5 = pointwise(relu(pointwise(layer_norm_channels(spatial_a5), W1_a5)), W2_a5)  # channel mixer with LayerNorm.
print("spatial shape:", spatial_a5.shape, "correction shape:", with_norm_corr_a5.shape)  # inspect intermediates.

In [ ]:
y_no_norm_a5 = x_a5[:4, :4, :] + no_norm_corr_a5  # residual output without normalization.
y_norm_a5 = x_a5[:4, :4, :] + with_norm_corr_a5  # residual output with normalization.
diff_a5 = np.mean(np.abs(y_norm_a5 - y_no_norm_a5))  # average ablation difference.
print("mean absolute output difference:", round(float(diff_a5), 4))  # inspect effect size.
assert diff_a5 > 0  # verify LayerNorm changes the computation.

In [ ]:
energy_spatial_a5 = float(np.mean(spatial_a5 ** 2))  # spatial-only energy.
energy_no_norm_a5 = float(np.mean(y_no_norm_a5 ** 2))  # residual energy without LN.
energy_norm_a5 = float(np.mean(y_norm_a5 ** 2))  # residual energy with LN.
print("energies:", np.round([energy_spatial_a5, energy_no_norm_a5, energy_norm_a5], 4))  # compare variants numerically.

In [ ]:
plt.figure(figsize=(7, 2.7))  # visualize channel-0 variants.
for idx_a5, (mat_a5, title_a5) in enumerate([(spatial_a5[:, :, 0], "spatial only"), (y_no_norm_a5[:, :, 0], "no LN residual"), (y_norm_a5[:, :, 0], "LN residual")]):
    plt.subplot(1, 3, idx_a5 + 1)  # one panel per variant.
    plt.imshow(mat_a5, cmap="viridis")  # show channel 0 response.
    plt.title(title_a5)  # label variant.
    plt.colorbar(fraction=.046)  # compact colorbar.
plt.suptitle("Advanced 5: block pieces change the signal"); plt.show()

▶ What you'll see: the variants produce visibly different channel-0 maps and different numeric energies.

👀 Takeaway: ConvNeXt is effective because its pieces are coordinated: local depthwise evidence, stable normalization, expressive channel mixing, and residual correction.

---

# Reference walkthrough — original compact notebook

The sections above build every idea from scratch with detailed steps and worked examples. Below is the original compact notebook for this lesson, kept as a concise reference and for its practice prompts.

ConvNeXt modernizes CNN blocks with depthwise spatial mixing, LayerNorm, inverted bottlenecks, and residual structure.

ConvNeXt keeps the convolutional inductive bias but borrows many training and block-design choices from modern vision transformers. A depthwise filter mixes space per channel, LayerNorm stabilizes channels, and pointwise layers expand then project. The result is still a CNN block, not attention.

Save a copy to Drive to edit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(7)


## The concept, built once (D1)
$$\mathrm{DWConv}(x)_c=K_c*x_c,\qquad \mathrm{hidden}=4C,\qquad \mathrm{pointwise}=C(4C)+(4C)C$$

We first write the reusable method and assert the exact lesson numbers before scaling to the ladder.

In [ ]:

def layer_norm_vector(v):
    mean = v.mean()
    variance = ((v - mean) ** 2).mean()
    normalized = (v - mean) / np.sqrt(variance)
    return mean, variance, normalized


def convnext_block(patch, channels=3, expansion=4):
    depthwise_kernel = np.array(
        [
            [1.0, 0.0, 0.0],
            [0.0, -1.0, 0.0],
            [0.0, 0.0, 0.0],
        ]
    )
    response = float((patch * depthwise_kernel).sum())
    hidden_channels = channels * expansion
    pointwise_weights = channels * hidden_channels + hidden_channels * channels
    return response, hidden_channels, pointwise_weights


lesson_patch = np.array(
    [
        [1.0, 0.0, 0.0],
        [0.0, 4.0, 0.0],
        [0.0, 0.0, 0.0],
    ]
)
response, hidden_channels, pointwise_weights = convnext_block(lesson_patch)
mean, variance, normalized = layer_norm_vector(np.array([1.0, 2.0, 3.0]))
ramp = 2.0 * np.add.outer(np.arange(5), np.arange(5))
ramp_responses = []
for i in range(3):
    for j in range(3):
        ramp_responses.append(convnext_block(ramp[i:i + 3, j:j + 3])[0])
ramp_responses = np.array(ramp_responses).reshape(3, 3)

assert response == -3.0
assert np.allclose(ramp_responses, -4.0)
assert mean == 2.0
assert np.isclose(variance, 2.0 / 3.0)
assert np.allclose(np.round(normalized, 3), [-1.225, 0.0, 1.225])
assert hidden_channels == 12
assert pointwise_weights == 72

print("depthwise response", response)
print("ramp responses")
print(ramp_responses)
print("LayerNorm", mean, variance, np.round(normalized, 3))
print("hidden channels", hidden_channels)
print("pointwise weights", pointwise_weights)


## Visual check
The numbers above are easier to trust when the intermediate feature behavior is visible.

In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(9, 3))

axes[0].imshow(lesson_patch, cmap="viridis")
axes[0].set_title("lesson patch")

axes[1].imshow(ramp_responses, cmap="coolwarm")
axes[1].set_title("ramp response")

axes[2].bar(["C", "4C", "weights"], [3, hidden_channels, pointwise_weights])
axes[2].set_title("bottleneck budget")

for ax in axes[:2]:
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

plt.tight_layout()
plt.show()


## Dataset ladder (D1 to D5)
We inline the shared CPU-safe classification ladder. Each rung returns images `X` with shape `(n, 8, 8)` and labels `y`, so the same featurizer can be evaluated from hand patches to the hardest fallback or cached MNIST rung.

In [ ]:
"""
F6 (Vision) shared dataset ladder — D1..D5 of rising complexity, CPU-only and run-all-safe.

This is the canonical ladder inlined into the classification-style Part-7 notebooks. Every
rung returns (X, y) with X shape (n, 8, 8) float in [0, 1] and integer labels y, so one
featurizer + classifier can run unchanged across all five rungs (the "watch it scale" story).

D4/D5 load real MNIST / CIFAR-10 via torchvision when the download is available (as in Colab),
offline they fall back to a harder synthetic set so run-all never fails. Code is written one
statement per line for readability.
"""

import numpy as np
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split


def _resize_to_8x8(img):
    """Nearest-neighbour resize of a 2-D array to 8x8 (no SciPy dependency)."""
    h, w = img.shape
    rows = (np.linspace(0, h - 1, 8)).round().astype(int)
    cols = (np.linspace(0, w - 1, 8)).round().astype(int)
    return img[np.ix_(rows, cols)]


def _normalize(x):
    """Scale an array into [0, 1], a flat array becomes all zeros."""
    x = x.astype(float)
    lo = x.min()
    hi = x.max()
    if hi - lo < 1e-12:
        return np.zeros_like(x)
    return (x - lo) / (hi - lo)


def d1_hand_patches():
    """D1 — hand-built 4x4 patches, 2 classes: a vertical line (col 1) vs a horizontal line (row 1).

    Fixed positions with light jitter, so the two classes are cleanly separable and the
    mechanism is fully visible — the easy first rung.
    """
    rng = np.random.default_rng(0)
    images = []
    labels = []
    for _ in range(24):
        patch = rng.uniform(0.0, 0.15, size=(4, 4))
        patch[:, 1] = rng.uniform(0.85, 1.0)
        images.append(_resize_to_8x8(patch))
        labels.append(0)
    for _ in range(24):
        patch = rng.uniform(0.0, 0.15, size=(4, 4))
        patch[1, :] = rng.uniform(0.85, 1.0)
        images.append(_resize_to_8x8(patch))
        labels.append(1)
    return np.array(images), np.array(labels)


def d2_synthetic_shapes():
    """D2 — clean synthetic shapes on an 8x8 grid, 2 classes (square vs disc)."""
    rng = np.random.default_rng(1)
    yy, xx = np.mgrid[0:8, 0:8]
    images = []
    labels = []
    for _ in range(80):
        img = rng.uniform(0.0, 0.1, size=(8, 8))
        img[2:6, 2:6] = 0.9
        images.append(img)
        labels.append(0)
    for _ in range(80):
        img = rng.uniform(0.0, 0.1, size=(8, 8))
        disc = (xx - 3.5) ** 2 + (yy - 3.5) ** 2 <= 4.0
        img[disc] = 0.9
        images.append(img)
        labels.append(1)
    return np.array(images), np.array(labels)


def d3_sklearn_digits():
    """D3 — real sklearn digits (native 8x8), 4 classes for a fast, honest multi-class rung."""
    digits = load_digits()
    keep = np.isin(digits.target, [0, 1, 2, 3])
    X = digits.images[keep]
    y = digits.target[keep]
    X = np.array([_normalize(img) for img in X])
    return X, y


def _synthetic_textured(n_per_class, n_classes, noise, seed):
    """A harder synthetic fallback: textured class prototypes at 8x8 with noise."""
    rng = np.random.default_rng(seed)
    protos = [rng.uniform(0.0, 1.0, size=(8, 8)) for _ in range(n_classes)]
    images = []
    labels = []
    for cls in range(n_classes):
        for _ in range(n_per_class):
            img = protos[cls] + rng.normal(0.0, noise, size=(8, 8))
            images.append(_normalize(img))
            labels.append(cls)
    return np.array(images), np.array(labels)


def _call_with_timeout(fn, seconds):
    """Run fn() but abort with TimeoutError after `seconds` (guards slow/hanging downloads)."""
    import signal

    def _raise(signum, frame):
        raise TimeoutError("download timed out")

    old = signal.signal(signal.SIGALRM, _raise)
    signal.alarm(seconds)
    try:
        return fn()
    finally:
        signal.alarm(0)
        signal.signal(signal.SIGALRM, old)


def _load_mnist_gray(classes, n_per_class, seed, shift=False, noise=0.0):
    """Load MNIST via torchvision, grayscale + resize to 8x8, subsample. Raises on failure.

    MNIST is a small (~11 MB) real dataset. CIFAR-10 is deliberately avoided (a 170 MB
    download breaks run-all-safety), the harder D5 rung instead shifts and noises MNIST.
    """
    import torchvision

    ds = torchvision.datasets.MNIST(root="./data", train=True, download=True)
    rng = np.random.default_rng(seed)
    targets = np.array(ds.targets)
    images = []
    labels = []
    for cls in classes:
        idx = np.where(targets == cls)[0][:n_per_class]
        for i in idx:
            arr = np.asarray(ds[int(i)][0], dtype=float)
            small = _resize_to_8x8(arr)
            if shift:
                small = np.roll(small, rng.integers(-1, 2), axis=0)
                small = np.roll(small, rng.integers(-1, 2), axis=1)
            if noise:
                small = small + rng.normal(0.0, noise * 255.0, size=(8, 8))
            images.append(_normalize(small))
            labels.append(cls)
    return np.array(images), np.array(labels)


def d4_mnist_or_fallback():
    """D4 — real MNIST (4 clean classes) when downloadable, else a harder synthetic set."""
    try:
        X, y = _call_with_timeout(lambda: _load_mnist_gray([0, 1, 2, 3], 60, seed=4), 30)
        return (X, y), "MNIST (real)"
    except Exception:
        return _synthetic_textured(60, 4, noise=0.35, seed=4), "synthetic (offline fallback)"


def d5_mnist_hard_or_fallback():
    """D5 — real MNIST, more classes with shift + noise (distribution shift), else hardest synthetic."""
    try:
        X, y = _call_with_timeout(lambda: _load_mnist_gray([0, 1, 2, 3, 4, 5], 60, seed=5, shift=True, noise=0.12), 30)
        return (X, y), "MNIST shifted+noisy (real, harder)"
    except Exception:
        return _synthetic_textured(60, 6, noise=0.6, seed=5), "synthetic (offline fallback)"


def load_ladder():
    """Return the five rungs as a list of (name, X, y). D4/D5 note whether real data loaded."""
    rungs = []
    rungs.append(("D1 hand patches", *d1_hand_patches()))
    rungs.append(("D2 synthetic shapes", *d2_synthetic_shapes()))
    rungs.append(("D3 sklearn digits", *d3_sklearn_digits()))
    (x4, y4), tag4 = d4_mnist_or_fallback()
    rungs.append((f"D4 {tag4}", x4, y4))
    (x5, y5), tag5 = d5_mnist_hard_or_fallback()
    rungs.append((f"D5 {tag5}", x5, y5))
    return rungs


def accuracy_with(featurize, X, y):
    """Map each image through featurize, train logistic regression, return held-out accuracy."""
    feats = np.array([featurize(img) for img in X])
    x_tr, x_te, y_tr, y_te = train_test_split(feats, y, test_size=0.4, random_state=0, stratify=y)
    clf = LogisticRegression(max_iter=2000)
    clf.fit(x_tr, y_tr)
    return clf.score(x_te, y_te)




rungs = load_ladder()

fig, axes = plt.subplots(1, 5, figsize=(12, 3))

for ax, (name, X, y) in zip(axes, rungs):
    ax.imshow(X[0], cmap="gray")
    ax.set_title(f"{name.split()[0]}\n{X.shape}\n{len(set(y.tolist()))} classes")
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

plt.tight_layout()
plt.show()

for name, X, y in rungs:
    print(f"{name:38s} X={X.shape} classes={sorted(set(y.tolist()))}")


## Run the same method across D1-D5
Only the data rung changes. The featurizer and accuracy metric stay fixed.

In [ ]:

def topic_feature_map(img):
    padded = np.pad(img, 1, mode="edge")
    mixed = np.zeros_like(img)
    kernel = np.array(
        [
            [1.0, 0.0, 0.0],
            [0.0, -1.0, 0.0],
            [0.0, 0.0, 0.0],
        ]
    )
    for i in range(img.shape[0]):
        for j in range(img.shape[1]):
            mixed[i, j] = (padded[i:i + 3, j:j + 3] * kernel).sum()
    normalized = (mixed - mixed.mean()) / (mixed.std() + 1e-12)
    residual = img + 0.1 * normalized
    return residual


def featurize(img):
    residual = topic_feature_map(img)
    return np.concatenate([img.ravel(), residual.ravel()])


accuracies = []

for name, X, y in rungs:
    acc = accuracy_with(featurize, X, y)
    accuracies.append(acc)
    print(f"{name:38s} accuracy={acc:.3f}")

baseline_accuracies = []

for name, X, y in rungs:
    acc = accuracy_with(lambda im: im.ravel(), X, y)
    baseline_accuracies.append(acc)

print("flat baseline", [round(x, 3) for x in baseline_accuracies])


## Results visualization
Top row: one feature or activation panel per rung. Bottom row: accuracy versus ladder complexity.

In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(14, 6))

for idx, (name, X, y) in enumerate(rungs):
    feature = topic_feature_map(X[0])
    if isinstance(feature, tuple):
        feature = feature[0]
    axes[0, idx].imshow(feature, cmap="viridis")
    axes[0, idx].set_title(name.split()[0])
    axes[0, idx].tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

axes[1, 0].plot(range(1, 6), accuracies, marker="o", label="topic features")
axes[1, 0].plot(range(1, 6), baseline_accuracies, marker="s", label="flat baseline")
axes[1, 0].set_xticks(range(1, 6))
axes[1, 0].set_xlabel("rung")
axes[1, 0].set_ylabel("accuracy")
axes[1, 0].set_ylim(0.0, 1.05)
axes[1, 0].legend()
axes[1, 0].set_title("accuracy vs rung")

for ax in axes[1, 1:]:
    ax.axis("off")

plt.tight_layout()
plt.show()


## Pitfall on D5: calling ConvNeXt attention
ConvNeXt is modern, but its spatial mixing is still convolution. Replacing depthwise convolution with full convolution or attention changes both cost and behavior, so we compare the counts and keep per-channel spatial filters.

In [ ]:

channels = 3
full_conv_weights = 7 * 7 * channels * channels
true_depthwise_weights = 7 * 7 * channels
attention_scores = 8 * 8 * 8 * 8

print("full conv weights", full_conv_weights)
print("depthwise weights", true_depthwise_weights)
print("attention score pairs", attention_scores)
print("ConvNeXt fix keeps per-channel spatial filters")


## Evaluate it + Practice
- Metric: held-out accuracy on every rung, compared with a no-skill flat-pixel logistic baseline.
- Sanity check: D1 should be easy enough to overfit or nearly overfit with the concept features.
- Ablation: remove the key block feature and accuracy should not improve over the flat baseline.
- Failure signal: the hardest rung may expose distribution shift, shape mismatch, or compute-budget mistakes before D1 does.

Practice prompts:
1. Change one design constant and rerun the accuracy curve.
2. Print the D5 confusion pattern for the worst two classes.
3. Replace the feature map panel with an example from a different class.

In [ ]:
# Your code here


In [ ]:
# Your code here


In [ ]:
# Your code here
